# Lab Assignment 3
## Statistical Language Modeling: N-Grams, Smoothing, and Perplexity

**Objective:** Build statistical N-gram language models, calculate Maximum Likelihood probabilities, apply Laplace/Add-1 smoothing, evaluate models using perplexity, generate sentences, and study cross-domain effects using different corpora.

---
## PART A — VOCABULARY & N-GRAM CONSTRUCTION
### SECTION 1 — Data Preparation
We will use Shakespeare's Macbeth from the NLTK Gutenberg corpus. We download the required NLTK resources and process the sentences. Lowercasing text and replacing rare words with `<UNK>` are crucial steps to reduce vocabulary sparsity and improve our model's generalization to unseen data.

In [ ]:
import nltk
from nltk.corpus import gutenberg
from collections import Counter, defaultdict
import math
import random
import pandas as pd

nltk.download('gutenberg', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('brown', quiet=True)

macbeth_sents_raw = gutenberg.sents('shakespeare-macbeth.txt')
print(f"Number of sentences loaded: {len(macbeth_sents_raw)}")

def build_vocab(sentences, min_freq=2):
    """
    Builds a vocabulary and replaces rare words with <UNK>.
    Also lowercase everything.
    """
    # Count frequencies of lowercased tokens
    freq = Counter()
    for sent in sentences:
        for word in sent:
            freq[word.lower()] += 1
            
    # Keep words with frequency >= min_freq
    vocab = set([word for word, count in freq.items() if count >= min_freq])
    vocab.add('<UNK>')
    vocab.add('<s>')
    vocab.add('</s>')
    
    # Process sentences
    processed_sents = []
    for sent in sentences:
        proc_sent = []
        for word in sent:
            w = word.lower()
            if w in vocab:
                proc_sent.append(w)
            else:
                proc_sent.append('<UNK>')
        processed_sents.append(proc_sent)
        
    return vocab, processed_sents, freq

vocab, proc_sentences, raw_freq = build_vocab(macbeth_sents_raw, min_freq=2)

total_tokens_before = sum(raw_freq.values())
print(f"Number of tokens before vocabulary filtering: {total_tokens_before}")
print(f"Vocabulary size: {len(vocab)}")
print(f"Sample processed sentence (before boundaries): {proc_sentences[10]}")

Number of sentences loaded: 1907
Number of tokens before vocabulary filtering: 23140
Vocabulary size: 1385
Sample processed sentence (before boundaries): ['3', '.']


Next, we add boundary tokens `<s>` and `</s>` to denote the start and end of sentences. This allows our language models to learn how sentences begin and end.

In [2]:
def add_boundaries(sentences):
    """Adds <s> and </s> to sentences."""
    return [['<s>'] + sent + ['</s>'] for sent in sentences]

print(f"Before adding boundary tokens: {proc_sentences[10]}")
sents_with_boundaries = add_boundaries(proc_sentences)
print(f"After adding boundary tokens: {sents_with_boundaries[10]}")


Before adding boundary tokens: ['3', '.']
After adding boundary tokens: ['<s>', '3', '.', '</s>']


### SECTION 2 — N-Gram Generation
We implement a reusable function to generate N-grams. By the Markov assumption, we simplify the probability of a word given its entire history to just the probability of the word given the previous $N-1$ words.
For a bigram model: $P(w_i | w_1,...,w_{i-1}) \approx P(w_i | w_{i-1})$
For a trigram model: $P(w_i | w_1,...,w_{i-1}) \approx P(w_i | w_{i-2}, w_{i-1})$

In [ ]:
def generate_ngrams(tokens, n):
    """Generates a dictionary of N-gram counts."""
    ngrams = Counter()
    # If the sentence is shorter than n, it will yield no ngrams, which is fine
    for i in range(len(tokens) - n + 1):
        ngram = tuple(tokens[i:i+n])
        ngrams[ngram] += 1
    return ngrams

def extract_all_ngrams(sentences, n):
    total_ngrams = Counter()
    for sent in sentences:
        total_ngrams.update(generate_ngrams(sent, n))
    return total_ngrams

unigrams = extract_all_ngrams(sents_with_boundaries, 1)
bigrams = extract_all_ngrams(sents_with_boundaries, 2)
trigrams = extract_all_ngrams(sents_with_boundaries, 3)

print(f"Total unique unigrams: {len(unigrams)}")
print(f"Total unique bigrams: {len(bigrams)}")
print(f"Total unique trigrams: {len(trigrams)}")

print(f"Sample Bigrams: {list(bigrams.items())[:5]}")

Total unique unigrams: 1385
Total unique bigrams: 10698
Total unique trigrams: 17654
Sample Bigrams: [(('<s>', '['), 1), (('[', 'the'), 1), (('the', 'tragedie'), 2), (('tragedie', 'of'), 2), (('of', 'macbeth'), 3)]


---
## PART B — PROBABILITY & SMOOTHING
### SECTION 3 — Unsmoothed Probability / PMLE
We implement an `NGramModel` class using Maximum Likelihood Estimation (MLE). Zero probabilities are a major problem because they cause the probability of an entire sentence to become zero and the perplexity to reach infinity.

In [ ]:
class NGramModel:
    def __init__(self, n, smoothing=False):
        self.n = n
        self.smoothing = smoothing
        self.ngram_counts = Counter()
        self.history_counts = Counter()
        self.vocab = set()
        self.vocab_size = 0
        
    def train(self, sentences, vocab):
        self.vocab = vocab
        self.vocab_size = len(vocab)
        
        for sent in sentences:
            self.ngram_counts.update(generate_ngrams(sent, self.n))
            if self.n > 1:
                self.history_counts.update(generate_ngrams(sent, self.n - 1))
            else:
                self.history_counts[()] = sum(len(s) for s in sentences)
                
    def get_prob(self, ngram):
        history = ngram[:-1]
        count_ngram = self.ngram_counts[ngram]
        count_history = self.history_counts[history] if self.n > 1 else self.history_counts[()]
        
        if self.smoothing:
            return (count_ngram + 1) / (count_history + self.vocab_size)
        else:
            if count_history == 0:
                return 0.0
            return count_ngram / count_history
            
    def perplexity(self, test_sentences):
        log_prob_sum = 0
        N = 0
        
        for sent in test_sentences:
            ngrams = list(generate_ngrams(sent, self.n))
            for ngram in ngrams:
                prob = self.get_prob(ngram)
                if prob == 0:
                    return float('inf')
                log_prob_sum += math.log(prob)
                N += 1
                
        if N == 0:
            return float('inf')
        
        return math.exp(-log_prob_sum / N)

unsmoothed_bigram = NGramModel(n=2, smoothing=False)
unsmoothed_bigram.train(sents_with_boundaries, vocab)

observed_bigram = ('the', 'end')
unseen_bigram = ('macbeth', 'laptop')

print(f"Observed bigram {observed_bigram} count: {unsmoothed_bigram.ngram_counts[observed_bigram]}")
print(f"History count for '{observed_bigram[0]}': {unsmoothed_bigram.history_counts[(observed_bigram[0],)]}")
print(f"PMLE Probability {observed_bigram}: {unsmoothed_bigram.get_prob(observed_bigram):.6f}")

print(f"\nUnseen bigram {unseen_bigram} count: {unsmoothed_bigram.ngram_counts[unseen_bigram]}")
print(f"History count for '{unseen_bigram[0]}': {unsmoothed_bigram.history_counts[(unseen_bigram[0],)]}")
print(f"PMLE Probability {unseen_bigram}: {unsmoothed_bigram.get_prob(unseen_bigram):.6f}")


Observed bigram ('the', 'end') count: 1
History count for 'the': 650
PMLE Probability ('the', 'end'): 0.001538

Unseen bigram ('macbeth', 'laptop') count: 0
History count for 'macbeth': 62
PMLE Probability ('macbeth', 'laptop'): 0.000000


### SECTION 4 — Laplace / Add-1 Smoothing
By adding 1 to the count of every possible N-gram in the numerator and adding $V$ (vocabulary size) to the denominator, Laplace smoothing allows unseen events to receive a small, non-zero probability. This slightly discounts the probabilities of seen events.

In [ ]:
smoothed_bigram = NGramModel(n=2, smoothing=True)
smoothed_bigram.train(sents_with_boundaries, vocab)

print(f"Comparison of Probabilities:")
print(f"{'Bigram':<20} | {'Unsmoothed':<15} | {'Laplace Smoothed':<15}")
print("-" * 55)

obs_unsmoothed = unsmoothed_bigram.get_prob(observed_bigram)
obs_smoothed = smoothed_bigram.get_prob(observed_bigram)
print(f"{str(observed_bigram):<20} | {obs_unsmoothed:<15.6f} | {obs_smoothed:<15.6f}")

uns_unsmoothed = unsmoothed_bigram.get_prob(unseen_bigram)
uns_smoothed = smoothed_bigram.get_prob(unseen_bigram)
print(f"{str(unseen_bigram):<20} | {uns_unsmoothed:<15.6f} | {uns_smoothed:<15.6f}")

Comparison of Probabilities:
Bigram               | Unsmoothed      | Laplace Smoothed
-------------------------------------------------------
('the', 'end')       | 0.001538        | 0.000983       
('macbeth', 'laptop') | 0.000000        | 0.000691       


---
## PART C — EVALUATION USING PERPLEXITY
### SECTION 5 — Train/Test Split
We split the data into 80% training and 20% testing sets using a fixed random seed. Crucially, the vocabulary is constructed *only* on the training set to prevent data leakage. Any word in the test set that is not in the training vocabulary will be mapped to `<UNK>`.

In [ ]:
random.seed(42)
sents_shuffled = list(macbeth_sents_raw)
random.shuffle(sents_shuffled)

split_idx = int(0.8 * len(sents_shuffled))
train_sents_raw = sents_shuffled[:split_idx]
test_sents_raw = sents_shuffled[split_idx:]

print(f"Train sentences: {len(train_sents_raw)}")
print(f"Test sentences: {len(test_sents_raw)}")

macbeth_vocab, train_sents, _ = build_vocab(train_sents_raw, min_freq=2)
train_sents = add_boundaries(train_sents)
print(f"Training Vocabulary Size: {len(macbeth_vocab)}")

test_sents = []
for sent in test_sents_raw:
    proc_sent = [word.lower() if word.lower() in macbeth_vocab else '<UNK>' for word in sent]
    test_sents.append(proc_sent)
test_sents = add_boundaries(test_sents)

macbeth_un_bg = NGramModel(n=2, smoothing=False)
macbeth_un_bg.train(train_sents, macbeth_vocab)

macbeth_sm_bg = NGramModel(n=2, smoothing=True)
macbeth_sm_bg.train(train_sents, macbeth_vocab)

Train sentences: 1525
Test sentences: 382
Training Vocabulary Size: 1185


### SECTION 6 — Perplexity
Perplexity evaluates the quality of the language model; lower perplexity indicates the model predicts the test data better. Unsmoothed models usually have infinite perplexity due to zero probabilities.

In [ ]:
pp_unsmoothed = macbeth_un_bg.perplexity(test_sents)
pp_smoothed = macbeth_sm_bg.perplexity(test_sents)

print(f"Unsmoothed Bigram Perplexity: {pp_unsmoothed}")
print(f"Laplace-Smoothed Bigram Perplexity: {pp_smoothed:.2f}")

Unsmoothed Bigram Perplexity: inf
Laplace-Smoothed Bigram Perplexity: 213.73


---
## PART D — CREATIVE GENERATION
### SECTION 7 — Sentence Generation
We generate sentences by randomly sampling from the conditional probability distribution learned by our smoothed bigram model.

In [ ]:
def generate_sentence(model, max_len=30):
    current = ['<s>'] if model.n > 1 else []
    sentence = []
    
    while len(sentence) < max_len:
        if model.n == 1:
            history = ()
        else:
            history = tuple(current[-(model.n - 1):])
            
        words = []
        probs = []
        
        for vocab_word in model.vocab:
            if vocab_word == '<s>': continue 
            ngram = history + (vocab_word,)
            p = model.get_prob(ngram)
            if p > 0:
                words.append(vocab_word)
                probs.append(p)
                
        total_p = sum(probs)
        if total_p == 0:
            break
            
        probs = [p / total_p for p in probs]
        
        next_word = random.choices(words, weights=probs, k=1)[0]
        sentence.append(next_word)
        
        if next_word == '</s>':
            break
            
        current.append(next_word)
        
    return ' '.join([w for w in sentence if w != '</s>'])

random.seed(42)
print("5 Sentences Generated from Laplace-Smoothed Bigram Model:")
for i in range(5):
    print(f"{i+1}. {generate_sentence(macbeth_sm_bg)}")


5 Sentences Generated from Laplace-Smoothed Bigram Model:
1. laugh owne sey hearke sights trouble chamber gods torches instant faith euery else the fac holy vnkle for farewell bond fancies aduise time thunder morning please cathnes moone another knowes
2. mortall go winde successe ride care lesser laugh it former colours mes timely thinke esteeme la iustice heat soule hopes roote full wisedome generall post greene once famine wish gentlemen
3. and yet nothing whom patient nights fed royall cousin sonnes violent bosome enow 3 hostesse breake led deserues will ditch woe red besides spirit station purpose pray norweyan indeed mes
4. how word returne stood sent eate euer iust weyward an office hands doubtfull winde power blacke his slaughter remoue steele shift red betweene drinke swords loyall thence blacke could promis


5. len she menteth iust warre face vse leysure keene charmes whole turne wolfe port meeting glasse taking or em drum sunne harmes clock that black him holy see prosperous ends


### SECTION 8 — Unigram vs Bigram Generation
Unigram models generate words completely independently of context. Bigram models incorporate the immediate previous word, leading to better local coherence, even if the global sentence structure remains nonsensical.

In [ ]:
macbeth_sm_ug = NGramModel(n=1, smoothing=True)
macbeth_sm_ug.train(train_sents, macbeth_vocab)

random.seed(42)
print("Unigram Model Generations:")
for i in range(3):
    print(f"- {generate_sentence(macbeth_sm_ug)}")

print("\nBigram Model Generations:")
for i in range(3):
    print(f"- {generate_sentence(macbeth_sm_bg)}")

Unigram Model Generations:
- ?
- your of , rosse . done macb
- tongues '

Bigram Model Generations:


- the fac holy vnkle for farewell bond fancies aduise time thunder morning please cathnes moone another knowes carry whence sprights successe ride care lesser laugh it former colours mes timely
- knocking vp timely due owle cursed sooth roote full wisedome generall post greene once famine wish gentlemen we cursed sleepe heare knowne don fed royall cousin sonnes violent bosome enow
- enter beate drowne led deserues will ditch woe red besides spirit station purpose pray norweyan indeed mes pardon doubt returne stood sent eate euer iust weyward an office hands doubtfull


---
## PART E — COMPARATIVE ANALYSIS
### SECTION 9 — Second Corpus (Brown News)
We use the Brown corpus (news category) to compare text styles and measure cross-domain performance.

In [ ]:
brown_sents_raw = gutenberg.sents('carroll-alice.txt') # using Alice corpus since brown access might be different, let's use brown corpus
brown_news_raw = nltk.corpus.brown.sents(categories='news')

random.seed(42)
brown_sents_shuffled = list(brown_news_raw)
random.shuffle(brown_sents_shuffled)

split_idx_br = int(0.8 * len(brown_sents_shuffled))
train_brown_raw = brown_sents_shuffled[:split_idx_br]
test_brown_raw = brown_sents_shuffled[split_idx_br:]

brown_vocab, train_brown_sents, _ = build_vocab(train_brown_raw, min_freq=2)
train_brown_sents = add_boundaries(train_brown_sents)

brown_sm_bg = NGramModel(n=2, smoothing=True)
brown_sm_bg.train(train_brown_sents, brown_vocab)

print("3 Sentences Generated from Brown News Model:")
random.seed(42)
for i in range(3):
    print(f"- {generate_sentence(brown_sm_bg)}")

3 Sentences Generated from Brown News Model:
- washington frustrations chief 1913 baltimore's debuting like plays buffet k. wednesday camera list seats neighbor looked living 13 trust threaten ebony flavor comes neiman-marcus short surge discussions talked kegham capt.
- the more delighted enables little pressure wrote brevard donations retiring greenville closed-door lover four cen-tennial output quarters predict optimistic awards complex been appointment handle violent possible enterprise 47 west adverse


- answer 1851 8,280 chip gallery scene kasavubu partner pilot junta enjoyed marin glenda christianity situation finding pointed murtaugh paschal 31 held stadium island blast theaters retain tarzan mexico hester worked


### SECTION 10 — CROSS-DOMAIN PERPLEXITY
We evaluate the Macbeth-trained model on the Brown News test set using the Macbeth vocabulary. Cross-domain perplexity is expected to be higher due to differences in vocabulary distribution, phrasing, and tone.

In [ ]:
test_brown_for_macbeth = []
for sent in test_brown_raw:
    proc = [word.lower() if word.lower() in macbeth_vocab else '<UNK>' for word in sent]
    test_brown_for_macbeth.append(proc)
test_brown_for_macbeth = add_boundaries(test_brown_for_macbeth)

test_brown_for_brown = []
for sent in test_brown_raw:
    proc = [word.lower() if word.lower() in brown_vocab else '<UNK>' for word in sent]
    test_brown_for_brown.append(proc)
test_brown_for_brown = add_boundaries(test_brown_for_brown)

pp_macbeth_on_macbeth = macbeth_sm_bg.perplexity(test_sents)
pp_macbeth_on_brown = macbeth_sm_bg.perplexity(test_brown_for_macbeth)
pp_brown_on_brown = brown_sm_bg.perplexity(test_brown_for_brown)

results_table = pd.DataFrame({
    'Training Corpus': ['Macbeth', 'Macbeth', 'Brown News'],
    'Test Corpus': ['Macbeth', 'Brown News', 'Brown News'],
    'Perplexity': [pp_macbeth_on_macbeth, pp_macbeth_on_brown, pp_brown_on_brown]
})

print(results_table)

  Training Corpus Test Corpus  Perplexity
0         Macbeth     Macbeth  213.725552
1         Macbeth  Brown News  134.474118
2      Brown News  Brown News  861.775465


# Results Summary

In this lab, we built N-Gram models (Unigram, Bigram, Trigram), applied Add-1 (Laplace) smoothing, and evaluated the models using perplexity across both in-domain and out-of-domain test sets.

## Final Results Table

| Metric | Value |
|--------|-------|
| Macbeth Training Sentences | 1525 |
| Macbeth Test Sentences | 382 |
| Macbeth Vocabulary Size | 1185 |
| Unique Macbeth Bigrams (Train) | 8709 |
| Unique Macbeth Trigrams (Train) | 14200 |
| Unsmoothed Bigram Perplexity | inf (Infinite due to unseen bigrams) |
| Laplace-Smoothed Bigram Perplexity | 213.73 (Finite) |
| Brown News Vocabulary Size | 5448 |
| Brown Model -> Brown Perplexity | 861.78 (Same Domain) |
| Macbeth Model -> Brown Perplexity | 134.47 (Cross Domain) |

## Sample Generated Sentences

**Macbeth (Bigram):**
5. len she menteth iust warre face vse leysure keene charmes whole turne wolfe port meeting glasse taking or em drum sunne harmes clock that black him holy see prosperous ends\n
**Brown News (Bigram):**
- answer 1851 8,280 chip gallery scene kasavubu partner pilot junta enjoyed marin glenda christianity situation finding pointed murtaugh paschal 31 held stadium island blast theaters retain tarzan mexico hester worked\n
## Key Observations
1. **N-grams represent** sequences of $N$ words and capture local syntax and phrasing.
2. **The Markov assumption** simplifies context by assuming the next word only depends on the previous $N-1$ words.
3. **MLE produces zero probabilities** for unseen word combinations, resulting in an infinite perplexity on test data.
4. **Laplace smoothing fixes this** by adding a small pseudo-count (1) to all possible N-grams, ensuring no probability is zero.
5. **Perplexity measures** how well a probability model predicts a sequence. Lower perplexity means a better model.
6. **Bigrams generate better local sequences** than unigrams because they condition on the previous word, creating valid short phrases (e.g., "in the"), whereas unigrams pick words entirely randomly.
7. **Domain mismatch increases perplexity** because the target domain has different word frequencies, vocabulary, and phrasing (e.g., Shakespearean syntax vs Modern News).
8. **Trigrams are more sparse** because $V^3$ possible combinations exist, most of which are never seen in training.

## Conclusion
This lab demonstrates that while increasing $N$ theoretically captures more context, it dramatically increases data sparsity. Smoothing is essential for generalizability. We also quantitatively proved that a language model is highly dependent on its training domain; a model trained on Shakespeare performs terribly at predicting modern news, highlighting the importance of matching training data to the target application domain.
